In [39]:
import pandas as pd
import numpy as np
import altair as alt
from sklearn.linear_model import LinearRegression

In [40]:
# Define paths to the individual CSV files

csv_files_output = {
    '50': 'emission_data/vllm_input_tok_summary/vLLM_50_word_summary.csv',
    '100': 'emission_data/vllm_input_tok_summary/vLLM_100_word_summary.csv',
    '250': 'emission_data/vllm_input_tok_summary/vLLM_250_word_summary.csv', 
    '500': 'emission_data/vllm_input_tok_summary/vLLM_500_word_summary.csv',
    '1000': 'emission_data/vllm_input_tok_summary/vLLM_1000_word_summary.csv',
    '2500': 'emission_data/vllm_input_tok_summary/vLLM_2500_word_summary.csv',
    '5000': 'emission_data/vllm_input_tok_summary/vLLM_5000_word_summary.csv',
    '7500': 'emission_data/vllm_input_tok_summary/vLLM_7500_word_summary.csv',
    # '10000': 'emission_data/vllm_input_tok_summary/vLLM_10000_word_summary.csv',
    # '15000': 'emission_data/vllm_input_tok_summary/vLLM_15000_word_summary.csv',
}

# Read the emissions data
emissions_data = pd.read_csv('input_tok_summary_vllm.csv')

In [41]:
# Initialize lists to store metadata
total_time = []
time_per_prompt = []
tok_per_sec = []
parameters_output = []
num_examples_output = []
num_prompts_output = []
total_emissions_output = []
cpu_energy_output = []
gpu_energy_output = []
ram_energy_output = []
total_energy_output = []
total_output_tokens_output = []
total_input_tokens_output = []
avg_input_tokens_output = []
avg_output_tokens_output = []

In [42]:
# Read and extract metadata from each CSV file
for model, file in csv_files_output.items():
    data = pd.read_csv(file)
    time = data.loc[data['Metric'] == 'Total Time', 'Value'].values[0]
    time_p_prompt = data.loc[data['Metric'] == 'AVG. Time / Prompt', 'Value'].values[0] / 1000 #Time is in ms
    tok_p_sec = data.loc[data['Metric'] == 'AVG. Tokens / Second', 'Value'].values[0]
    prompts = data.loc[data['Metric'] == 'Total Prompts', 'Value'].values[0]
    output_tokens = data.loc[data['Metric'] == 'Total Output Tokens', 'Value'].values[0]
    input_tokens = data.loc[data['Metric'] == 'Total Input Tokens', 'Value'].values[0]
    avg_i_tok = data.loc[data['Metric'] == 'AVG. Input Tokens / Prompt', 'Value'].values[0]
    avg_o_tok = data.loc[data['Metric'] == 'AVG. Output Tokens / Prompt', 'Value'].values[0]
    total_time.append(float(time))
    time_per_prompt.append(float(time_p_prompt))
    tok_per_sec.append(float(tok_p_sec))
    parameters_output.append(8)
    num_examples_output.append(int(model))
    num_prompts_output.append(int(prompts))
    total_output_tokens_output.append(float(output_tokens))
    total_input_tokens_output.append(float(input_tokens))
    avg_input_tokens_output.append(float(avg_i_tok))
    avg_output_tokens_output.append(float(avg_o_tok))    

In [43]:
# Extract emissions data
for model in csv_files_output.keys():
    model_emissions = emissions_data[emissions_data['project_name'].str.contains("vLLM_" + model + "_word_summary")]
    total_emissions_output.append(model_emissions['emissions'].values[0])
    cpu_energy_output.append(model_emissions['cpu_energy'].values[0])
    gpu_energy_output.append(model_emissions['gpu_energy'].values[0])
    ram_energy_output.append(model_emissions['ram_energy'].values[0])
    total_energy_output.append(model_emissions['energy_consumed'].values[0])


In [44]:
print(avg_input_tokens_output)
print(avg_output_tokens_output)
print(total_output_tokens_output)
print(total_emissions_output)

[441.0, 748.0, 1282.0, 2220.0, 3885.0, 7759.0, 14922.0, 25457.0]
[96.998, 152.015, 217.437, 285.218, 353.563, 427.407, 490.242, 567.433]
[96998.0, 152015.0, 217437.0, 285218.0, 353563.0, 427407.0, 490242.0, 567433.0]
[0.0056505383670293, 0.0070889279248674, 0.011731848486881, 0.0193749162103813, 0.0324112581576919, 0.075691368856632, 0.1424516719035163, 0.2168158357426798]


In [45]:
# Prepare data for regression and visualization
total_time = np.array(total_time)
time_per_prompt = np.array(time_per_prompt)
tok_per_sec = np.array(tok_per_sec)
parameters_output = np.array(parameters_output)
num_examples_output = np.array(num_examples_output)
num_prompts_output = np.array(num_prompts_output)
total_output_tokens_output = np.array(total_output_tokens_output)
total_input_tokens_output = np.array(total_input_tokens_output)
avg_input_tokens_output = np.array(avg_input_tokens_output)
avg_output_tokens_output = np.array(avg_output_tokens_output)
total_emissions_output = np.array(total_emissions_output)
cpu_energy_output = np.array(cpu_energy_output)
gpu_energy_output = np.array(gpu_energy_output)
ram_energy_output = np.array(ram_energy_output)
total_energy_output = np.array(total_energy_output)

In [46]:
print(total_time)

[  74.64758801   92.15549588  149.13437462  244.16428804  416.35407543
  957.70009375 1774.5068748  2685.21677136]


In [47]:
idle_gpu_power = 28*4 # 28W per GPU, 4 GPUs

total_idle_gpu_energy = (idle_gpu_power/1000)*(total_time/3600) # Convert W into kw and s into h
idle_gpu_energy_per_thousand_prompts = total_idle_gpu_energy / avg_output_tokens_output * 100_000 * 1000,

gpu_energy_without_idle = gpu_energy_output - total_idle_gpu_energy
gpu_energy_without_idle_per_thousand_prompts = gpu_energy_without_idle / avg_output_tokens_output * 100_000 * 1000,

In [48]:
# Calculate emissions per 10,000 prompts
emissions_per_thousand_prompts = {
    'Total Emissions per 100.000 output tokens': total_emissions_output / avg_output_tokens_output * 100_000 * 1000,
    'CPU Energy per 100.000 output tokens': cpu_energy_output / avg_output_tokens_output * 100_000 * 1000,
    'GPU Energy per 100.000 output tokens': gpu_energy_output / avg_output_tokens_output * 100_000 * 1000,
    'GPU Energy per 100.000 output tokens (without idle)': np.concatenate([gpu_energy_without_idle_per_thousand_prompts]).flatten(),
    'GPU Energy per 100.000 output tokens (idle)': np.concatenate([idle_gpu_energy_per_thousand_prompts]).flatten(),
    'RAM Energy per 100.000 output tokens': ram_energy_output / avg_output_tokens_output * 100_000 * 1000,
    'Total Energy per 100.000 output tokens': total_energy_output / avg_output_tokens_output * 100_000 * 1000,
}

In [49]:
print(f"GPU Energy per 100.000 output tokens: {emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens']}")
print(f"Idle GPU Energy per 100.000 output tokens: {emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens (idle)']}")
print(f"GPU Energy without idle per 100.000 output tokens: {emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens (without idle)']}")

GPU Energy per 100.000 output tokens: [ 5875.41001796  4740.46557063  5543.0556784   7007.09818242
  9372.21276403 18234.54556161 30132.18544192 39726.36048125]
Idle GPU Energy per 100.000 output tokens: [ 2394.24462865  1886.03747781  2133.83007457  2663.30396213
  3663.62936798  6971.13384377 11261.14868765 14722.45663036]
GPU Energy without idle per 100.000 output tokens: [ 3481.16538931  2854.42809282  3409.22560383  4343.7942203
  5708.58339604 11263.41171784 18871.03675427 25003.90385089]


In [50]:
print(emissions_per_thousand_prompts)

{'Total Emissions per 100.000 output tokens': array([ 5825.4173973 ,  4663.30817674,  5395.51616647,  6793.02014963,
        9167.03901644, 17709.43593732, 29057.41896931, 38209.94474108]), 'CPU Energy per 100.000 output tokens': array([1108.71066323,  873.37647209,  988.01630679, 1233.10586436,
       1696.18297523, 3227.37009078, 5213.44116651, 6815.84134583]), 'GPU Energy per 100.000 output tokens': array([ 5875.41001796,  4740.46557063,  5543.0556784 ,  7007.09818242,
        9372.21276403, 18234.54556161, 30132.18544192, 39726.36048125]), 'GPU Energy per 100.000 output tokens (without idle)': array([ 3481.16538931,  2854.42809282,  3409.22560383,  4343.7942203 ,
        5708.58339604, 11263.41171784, 18871.03675427, 25003.90385089]), 'GPU Energy per 100.000 output tokens (idle)': array([ 2394.24462865,  1886.03747781,  2133.83007457,  2663.30396213,
        3663.62936798,  6971.13384377, 11261.14868765, 14722.45663036]), 'RAM Energy per 100.000 output tokens': array([ 1776.5869614

In [51]:
print(parameters_output)

[8 8 8 8 8 8 8 8]


In [52]:
# Define the test types and model types
test_types = ['Input-tok-vllm']
model_types = ['llama3']

# Define the parameters
parameters = np.concatenate([parameters_output])
num_examples = np.concatenate([num_examples_output])
num_prompts = np.concatenate([num_prompts_output])
total_out_tok = np.concatenate([total_output_tokens_output])
total_in_tok = np.concatenate([total_input_tokens_output])
avg_out_tok = np.concatenate([avg_output_tokens_output])
avg_in_tok = np.concatenate([avg_input_tokens_output])

actual_emissions_per_100k_output_tokens = emissions_per_thousand_prompts['Total Emissions per 100.000 output tokens']

actual_cpu_energy_per_100k_output_tokens = emissions_per_thousand_prompts['CPU Energy per 100.000 output tokens']

actual_gpu_energy_per_100k_output_tokens = emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens']

actual_ram_energy_per_100k_output_tokens = emissions_per_thousand_prompts['RAM Energy per 100.000 output tokens']

actual_total_energy_per_100k_output_tokens = emissions_per_thousand_prompts['Total Energy per 100.000 output tokens']

actual_idle_gpu_energy_per_100k_output_tokens = emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens (idle)']

actual_non_idle_gpu_energy_per_100k_output_tokens = emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens (without idle)']

In [53]:
# Repeat test types and model types for each data point
test_type_column = np.concatenate([
    np.repeat(test_types[0], len(parameters_output))
])

model_type_column = np.concatenate([
    np.repeat(model_types[0], len(parameters_output))
])

In [54]:
print(len(actual_emissions_per_100k_output_tokens))
print(len(actual_cpu_energy_per_100k_output_tokens))
print(len(actual_gpu_energy_per_100k_output_tokens))
print(len(actual_ram_energy_per_100k_output_tokens))
print(len(actual_total_energy_per_100k_output_tokens))
print(len(actual_idle_gpu_energy_per_100k_output_tokens))
print(len(actual_non_idle_gpu_energy_per_100k_output_tokens))
print(len(test_type_column))
print(len(model_type_column))


8
8
8
8
8
8
8
8
8


In [55]:
# Create the dataframe
df = pd.DataFrame({
    'test_type': test_type_column,
    'model_type': model_type_column,
    'parameters': parameters,
    'num_examples': num_examples,
    'num_prompts': num_prompts,
    'total_time': total_time,
    'time_per_prompt': time_per_prompt,
    'tok_per_sec': tok_per_sec,
    'total_out_tok': total_out_tok,
    'total_in_tok': total_in_tok,
    'avg_out_tok': avg_out_tok,
    'avg_in_tok': avg_in_tok,
    'actual_emissions_per_100k_output_tokens': actual_emissions_per_100k_output_tokens,
    'actual_total_energy_per_100k_output_tokens': actual_total_energy_per_100k_output_tokens,
    'actual_cpu_energy_per_100k_output_tokens': actual_cpu_energy_per_100k_output_tokens,
    'actual_gpu_energy_per_100k_output_tokens': actual_gpu_energy_per_100k_output_tokens,
    'actual_ram_energy_per_100k_output_tokens': actual_ram_energy_per_100k_output_tokens,
    'actual_idle_gpu_energy_per_100k_output_tokens': actual_idle_gpu_energy_per_100k_output_tokens,
    'actual_non_idle_gpu_energy_per_100k_output_tokens': actual_non_idle_gpu_energy_per_100k_output_tokens,
})

df

,test_type,model_type,parameters,num_examples,num_prompts,total_time,time_per_prompt,tok_per_sec,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,actual_emissions_per_100k_output_tokens,actual_total_energy_per_100k_output_tokens,actual_cpu_energy_per_100k_output_tokens,actual_gpu_energy_per_100k_output_tokens,actual_ram_energy_per_100k_output_tokens,actual_idle_gpu_energy_per_100k_output_tokens,actual_non_idle_gpu_energy_per_100k_output_tokens
0,Input-tok-vllm,llama3,8,50,10000,74.647588,0.074648,1299.412380,96998.0,441000.0,96.998,441.0,5825.417397,8760.707643,1108.710663,5875.410018,1776.586961,2394.244629,3481.165389
1,Input-tok-vllm,llama3,8,100,10000,92.155496,0.092155,1649.548934,152015.0,748000.0,152.015,748.0,4663.308177,7013.039032,873.376472,4740.465571,1399.196989,1886.037478,2854.428093
2,Input-tok-vllm,llama3,8,250,10000,149.134375,0.149134,1457.993843,217437.0,1282000.0,217.437,1282.0,5395.516166,8114.189335,988.016307,5543.055678,1583.117350,2133.830075,3409.225604
3,Input-tok-vllm,llama3,8,500,10000,244.164288,0.244164,1168.139707,285218.0,2220000.0,285.218,2220.0,6793.020150,10215.862570,1233.105864,7007.098182,1975.658523,2663.303962,4343.794220
4,Input-tok-vllm,llama3,8,1000,10000,416.354075,0.416354,849.188277,353563.0,3885000.0,353.563,3885.0,9167.039016,13786.093475,1696.182975,9372.212764,2717.697736,3663.629368,5708.583396
5,Input-tok-vllm,llama3,8,2500,10000,957.700094,0.957700,446.284805,427407.0,7759000.0,427.407,7759.0,17709.435937,26632.802455,3227.370091,18234.545562,5170.886803,6971.133844,11263.411718
6,Input-tok-vllm,llama3,8,5000,10000,1774.506875,1.774507,276.269428,490242.0,14922000.0,490.242,14922.0,29057.418969,43698.766128,5213.441167,30132.185442,8353.139519,11261.148688,18871.036754
7,Input-tok-vllm,llama3,8,7500,10000,2685.216771,2.685217,211.317390,567433.0,25457000.0,567.433,25457.0,38209.944741,57463.033477,6815.841346,39726.360481,10920.831650,14722.456630,25003.903851


In [56]:
# Define chart width and height
chart_width = 1000
chart_height = 700

x_title = 'Average Input Tokens per Prompt'
x_data = 'avg_in_tok'
test_type = 'Input-tok-vllm'

scatter = alt.Chart(df).mark_circle(size=100).encode(
    x=alt.X(x_data, title=x_title),
    y=alt.Y('actual_total_energy_per_100k_output_tokens', title='Energy Consumption per 100.000 output tokens (in Wh)'),
    tooltip=[
        alt.Tooltip('parameters', title='Parameters (billions)'),
        alt.Tooltip('actual_emissions_per_100k_output_tokens', title='Actual Emissions per 100.000 output tokens'),
        alt.Tooltip('avg_out_tok', title='Average Output Tokens per Prompt'),
        alt.Tooltip('avg_in_tok', title='Average Input Tokens per Prompt'),
        alt.Tooltip('num_examples', title='Number of Examples'),
        alt.Tooltip('num_prompts', title='Number of Prompts'),
        alt.Tooltip('model_type', title='Model Type'),
        alt.Tooltip('test_type', title='Test Type'),
        alt.Tooltip('test_type', title='Test Type'),
    ]
).properties(
    title=f'Energy per 100.000 Output Tokens',
    width=chart_width,
    height=chart_height
)

# Create line plots for predicted emissions
line = scatter.transform_regression(x_data, 'actual_total_energy_per_100k_output_tokens', method="linear").mark_line()

# Create a combined chart with overlays for all emission types
combined_chart = alt.layer(scatter+line).resolve_scale(
    x='independent'
).properties(
    title='Energy per 100.000 output tokens',
    width=chart_width,  # Adjusted width for combined chart
    height=chart_height  # Adjusted height for combined chart
)

combined_chart.show()

alt.LayerChart(...)

In [57]:
# Store the results in a CSV file
df.to_csv('../results/data/input_tok_summary_vllm.csv', index=False)